# Preprocessing for Short-Term Electricity Demand Forecasting
This notebook loads and cleans power, weather, and economic datasets, merges them into a single hourly training table, engineers lag features, creates `next_hour_demand_mw`, and writes `preprocessed_data.csv`.

In [ ]:
import pandas as pd

power_path = '../data/PGCB_date_power_demand.xlsx'
weather_path = '../data/weather_data.xlsx'
econ_path = '../data/economic_full_1.csv'

power_df = pd.read_excel(power_path)
power_df['datetime'] = pd.to_datetime(power_df['datetime']).sort_values()
power_df = power_df.dropna(subset=['datetime'])
power_df = power_df.drop_duplicates(subset=['datetime'])

#fills in missing hours with NaN, which we can clean later.
power_df = power_df.set_index('datetime').asfreq('h').reset_index()

#XGBoost can't read text columns, so we can drop 'remarks' if it exists
if 'remarks' in power_df.columns:
    power_df = power_df.drop(columns=['remarks'])

empty_fraction = {}
for col in power_df.columns:
    empty_fraction[col] = power_df[col].isna().mean()

# dropping useless columns( > 80% missing)
drop_cols = []
for col, frac in empty_fraction.items():
    if frac > 0.8:
        drop_cols.append(col)
power_df = power_df.drop(columns=drop_cols)

generation_columns = []
for c in power_df.columns:
    if c in ['generation_mw','gas','liquid_fuel','coal','hydro','solar','wind']:
        generation_columns.append(c)

for col in ['demand_mw'] + generation_columns:
    if col in power_df.columns:
        power_df[col] = pd.to_numeric(power_df[col])

# limits outliers in demand_mw
if 'demand_mw' in power_df.columns:
    lower = power_df['demand_mw'].quantile(0.01)
    upper = power_df['demand_mw'].quantile(0.99)
    power_df['demand_mw'] = power_df['demand_mw'].clip(lower=lower, upper=upper)

numeric_cols = power_df.select_dtypes(include=['number']).columns.tolist()

power_df[numeric_cols] = power_df[numeric_cols].ffill(limit=3)
power_df[numeric_cols] = power_df[numeric_cols].interpolate(method='linear', limit_direction='both')

power_df['Year'] = power_df['datetime'].dt.year

print('Power dataset shape after cleaning:', power_df.shape)
print('Dropped columns with >80% missing values:', drop_cols)

Power dataset shape after cleaning: (89101, 13)
Dropped columns with >80% missing values: ['india_adani', 'nepal']


In [3]:
weather_df = pd.read_excel(weather_path, skiprows=3)

weather_df = weather_df.rename(columns={
    'time': 'datetime',
    'temperature_2m (°C)': 'temperature',
    'relative_humidity_2m (%)': 'humidity',
    'precipitation (mm)': 'precipitation'
})

weather_df['datetime'] = pd.to_datetime(weather_df['datetime'], errors='coerce')
weather_df = weather_df.dropna(subset=['datetime'])

for col in ['temperature', 'humidity', 'precipitation']:
    if col in weather_df.columns:
        weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')

existing_weather_cols = []
for c in ['temperature', 'humidity', 'precipitation']:
    if c in weather_df.columns:
        existing_weather_cols.append(c)

weather_df[existing_weather_cols] = weather_df[existing_weather_cols].interpolate(method='linear', limit_direction='both')
weather_df = weather_df[['datetime'] + existing_weather_cols].drop_duplicates(subset=['datetime'])

# Merging this with power data
merged_df = power_df.merge(weather_df, on='datetime', how='left')
print('Shape after merging weather data:', merged_df.shape)

Shape after merging weather data: (89101, 16)


In [20]:
econ_df = pd.read_csv(econ_path)
econ_df = econ_df.dropna(how='all').dropna(axis=1, how='all')

year_cols = []
for c in econ_df.columns:
    if c.isdigit():
        year_int = int(c)
        if 2015 <= year_int <= 2025:
            year_cols.append(c)

econ_long = econ_df.melt(
    id_vars=['Country Name', 'Indicator Name', 'Indicator Code'],
    value_vars=year_cols,
    var_name='Year',
    value_name='value'
)

econ_long['Year'] = pd.to_numeric(econ_long['Year'], errors='coerce').astype('Int64')
econ_long = econ_long.dropna(subset=['Year'])
econ_long['Year'] = econ_long['Year'].astype(int)
econ_long['value'] = pd.to_numeric(econ_long['value'], errors='coerce')

econ_pivot = econ_long.pivot_table(
    index='Year',
    columns='Indicator Name',
    values='value',
    aggfunc='mean'
).sort_index()

useful_indicators = [
    'GDP per capita (current US$)',
    'GDP growth (annual %)',
    'Inflation, consumer prices (annual %)',
    'Population, total',
    'Fossil fuel energy consumption (% of total)',
    'Renewable energy consumption (% of total final energy consumption)'
]

existing_useful = [col for col in useful_indicators if col in econ_pivot.columns]
econ_pivot = econ_pivot[existing_useful]

econ_pivot = econ_pivot.interpolate(method='linear', axis=0, limit_direction='both')
econ_pivot = econ_pivot.ffill().bfill()

# And finally we merge all files together
merged_df = merged_df.merge(econ_pivot.reset_index(), on='Year', how='left')
print('Shape after merging economic data:', merged_df.shape)

Shape after merging economic data: (89101, 22)


In [21]:
merged_df['hour'] = merged_df['datetime'].dt.hour
merged_df['dayofweek'] = merged_df['datetime'].dt.dayofweek
merged_df['month'] = merged_df['datetime'].dt.month
merged_df['day'] = merged_df['datetime'].dt.day

for lag in [1, 24, 168]:
    merged_df[f'lag_{lag}'] = merged_df['demand_mw'].shift(lag)
merged_df['next_hour_demand_mw'] = merged_df['demand_mw'].shift(-1)

lag_cols = []
for lag in [1, 24, 168]:
    lag_cols.append(f'lag_{lag}')

required_cols = lag_cols + ['next_hour_demand_mw']
merged_df = merged_df.dropna(subset=[c for c in required_cols if c in merged_df.columns]).reset_index(drop=True)

output_path = 'preprocessed_data.csv'
merged_df.to_csv(output_path, index=False)
print('Final preprocessed dataset shape:', merged_df.shape)
print('Saved preprocessed_data.csv to', output_path)
print('Columns saved:', merged_df.columns.tolist())

Final preprocessed dataset shape: (88932, 30)
Saved preprocessed_data.csv to preprocessed_data.csv
Columns saved: ['datetime', 'generation_mw', 'demand_mw', 'load_shedding', 'gas', 'liquid_fuel', 'coal', 'hydro', 'solar', 'wind', 'india_bheramara_hvdc', 'india_tripura', 'Year', 'temperature', 'humidity', 'precipitation', 'GDP per capita (current US$)', 'GDP growth (annual %)', 'Inflation, consumer prices (annual %)', 'Population, total', 'Fossil fuel energy consumption (% of total)', 'Renewable energy consumption (% of total final energy consumption)', 'hour', 'dayofweek', 'month', 'day', 'lag_1', 'lag_24', 'lag_168', 'next_hour_demand_mw']
